# Bronze Layer: Raw Listings Ingestion

This notebook implements the **Bronze layer** of the medallion architecture for Iasi housing market data.

## Purpose
The Bronze layer stores **raw, unprocessed data** exactly as scraped from Storia.ro:
* No transformations or cleaning
* Preserves original data format and quality
* Serves as a single source of truth
* Enables reprocessing if downstream logic changes

## Data Source
* **Source**: Storia.ro apartment listings (Iasi, Romania)
* **Method**: Web scraping using BeautifulSoup
* **Storage**: Delta Lake table in `bronze.raw_listings`

## Fields Captured
* `title_raw`: Raw listing title text
* `price_raw`: Raw price text (typically in €)
* `description_raw`: Raw description/details
* `url`: Listing URL
* `scraped_at`: Timestamp of scraping

In [0]:
%pip install -r /Workspace/Repos/bejmafei@gmail.com/iasi-housing-market-analysis/requirements.txt

In [0]:
# Parameterized configuration for scraping
dbutils.widgets.text("start_page", "1", "Start Page")
dbutils.widgets.text("end_page", "10", "End Page")
dbutils.widgets.dropdown("write_mode", "append", ["append", "overwrite"], "Write Mode")

# Get parameter values
start_page = int(dbutils.widgets.get("start_page"))
end_page = int(dbutils.widgets.get("end_page"))
write_mode = dbutils.widgets.get("write_mode")

print(f"Scraping pages {start_page} to {end_page}")
print(f"Write mode: {write_mode}")

In [0]:
# Import the scraper from the Repos directory
import sys
sys.path.append("/Workspace/Repos/bejmafei@gmail.com/iasi-housing-market-analysis/pipeline/scraper")

from scraper import scrape_page
import pandas as pd
from datetime import datetime

print("Scraper imported successfully")

In [0]:
%sql
-- Create the bronze schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Bronze layer: Raw, unprocessed data from source systems'

In [0]:
# Scrape data from specified page range
all_records = []

for page_num in range(start_page, end_page + 1):
    print(f"Scraping page {page_num}...")
    try:
        records = scrape_page(page_num)
        all_records.extend(records)
        print(f"Extracted {len(records)} listings from page {page_num}")
    except Exception as e:
        print(f"Error on page {page_num}: {e}")
        continue

print(f"\nTotal records scraped: {len(all_records)}")

# Convert to Spark DataFrame
if all_records:
    # Convert to pandas first for easier handling
    df_pandas = pd.DataFrame(all_records)
    
    # Convert to Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # Write to Bronze Delta table
    df_spark.write \
        .format("delta") \
        .mode(write_mode) \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze.raw_listings")
    
    print(f"✓ Successfully wrote {len(all_records)} records to bronze.raw_listings")
    display(df_spark.limit(5))
else:
    print("⚠ No records scraped. Check your page range or scraper configuration.")

In [0]:
%sql
-- Verify the data was written successfully
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT url) as unique_listings,
    MIN(scraped_at) as earliest_scrape,
    MAX(scraped_at) as latest_scrape
FROM bronze.raw_listings

## Next Steps

Now that raw data is in the Bronze layer:

1. **Silver Layer**: Create transformation logic to:
   * Parse price values from text (remove €, convert to numeric)
   * Extract structured information from titles (rooms, square meters, etc.)
   * Clean and standardize descriptions
   * Deduplicate listings by URL
   * Add data quality checks

2. **Gold Layer**: Build aggregated, business-ready tables:
   * Price trends by neighborhood
   * Average price per square meter
   * Listing volume over time
   * Property features analysis